# SuperLLM 스케일업 — 30만개 SFT (자립형 Colab)

**LFM2.5-350M을 30만개 고품질 instruct 데이터로 full fine-tuning.**

온라인 ULD(교사 매 스텝 forward)는 30만개면 100시간+라 비현실적. 대신 **이미
GPT-4로 생성된 고품질 데이터(OpenHermes-2.5)로 SFT** = sequence-level 증류.
배치·패킹으로 ~2-4시간. Gemma 교사·16GB 다운로드·gated 로그인 전부 불필요.

> 런타임 → 런타임 유형 변경 → **T4 GPU**. 세션 끊기면 Step 2·5만 다시 → 이어짐.

## Step 1 — 설치

In [ ]:
!pip -q install -U transformers trl peft datasets accelerate bitsandbytes

## Step 2 — Drive 마운트 (매 세션 실행)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3 — 데이터 30만개 준비 (OpenHermes-2.5 → messages)
GPT-4로 생성된 고품질 대화 데이터. ShareGPT 형식을 chat messages로 변환.

In [ ]:
import os, json
OUT  = '/content/drive/MyDrive/superllm_sft/lfm2.5-350m-300k'
DATA = '/content/drive/MyDrive/superllm_sft/openhermes_300k.jsonl'
os.makedirs(os.path.dirname(DATA), exist_ok=True)

if not os.path.exists(DATA):
    from datasets import load_dataset
    ds = load_dataset('teknium/OpenHermes-2.5', split='train').select(range(300000))
    role = {'human':'user', 'gpt':'assistant', 'system':'system'}
    n = 0
    with open(DATA, 'w') as f:
        for ex in ds:
            msgs = [{'role': role.get(t['from'],'user'), 'content': t['value']}
                    for t in ex['conversations']]
            if len(msgs) >= 2:
                f.write(json.dumps({'messages': msgs}, ensure_ascii=False) + '\n')
                n += 1
    print('wrote', n, '→', DATA)
else:
    print('already exists →', DATA)

## Step 4 — 모델·토크나이저 로드 (full fine-tuning)
350M은 작아서 T4에서 full FT 가능(LoRA보다 규모 데이터에 유리).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
BASE = 'LiquidAI/LFM2.5-350M'
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.bfloat16, attn_implementation='eager')
model.config.use_cache = False

## Step 5 — SFT 학습  ⭐ 매 세션 이 셀 재실행 = 이어하기

Drive에 체크포인트 저장(save_steps). 세션 끊기면 Step 2·4 재실행 후 이 셀 다시 →
checkpoint에서 자동 재개. 30만개 1 epoch ≈ 2-4시간(T4).

In [ ]:
import glob
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

ds = load_dataset('json', data_files=DATA, split='train')
cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    bf16=True,
    packing=True,
    max_length=1024,
    gradient_checkpointing=True,
    assistant_only_loss=True,   # 손실은 assistant 토큰만
    logging_steps=20,
    save_steps=500,
    save_total_limit=2,
    report_to='none',
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok)
resume = bool(glob.glob(os.path.join(OUT, 'checkpoint-*')))
print('resume from checkpoint:', resume)
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('done →', OUT)

## Step 6 — GGUF q4_k_m 변환 (학습 완료 후)

full FT라 병합 불필요 — 저장된 모델을 바로 변환. (llama.cpp 빌드는 `-j 2`로 OOM 방지,
transformers 최신이면 토크나이저 OK.)

In [ ]:
# llama.cpp 빌드 (한 번만)
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp
    !pip -q install -r llama.cpp/requirements.txt
    !pip -q install -U transformers tokenizers   # 변환용 최신 유지
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF > /tmp/cm.log 2>&1
!cmake --build llama.cpp/build --target llama-quantize llama-cli -j 2
!ls -la llama.cpp/build/bin/llama-quantize

In [ ]:
GGUF = '/content/drive/MyDrive/superllm_sft/lfm2.5-350m-300k-q4_k_m.gguf'
!python llama.cpp/convert_hf_to_gguf.py {OUT} --outfile /content/m-f16.gguf --outtype f16
!./llama.cpp/build/bin/llama-quantize /content/m-f16.gguf {GGUF} Q4_K_M
import os; print('최종:', GGUF, f'{os.path.getsize(GGUF)/1e6:.0f} MB')

## Step 7 — 실행 테스트

In [ ]:
!./llama.cpp/build/bin/llama-cli -m /content/drive/MyDrive/superllm_sft/lfm2.5-350m-300k-q4_k_m.gguf \
    -p "Explain quantum entanglement simply.\n" -n 200 -no-cnv